# CM3070 Final Project
# Model 3 - LLM-Based Explanation Generation

This notebook implements **Model 3** of the Contract Analysis AI pipeline: it takes clauses already classified by Model 2 (clause type + confidence) and generates a **clause-specific**, plain-English explanation and risk assessment for the employee.

**Why this is needed on top of Model 2's existing risk logic:** Model 2's `RISK_MAP` gives the exact same generic reason to every clause of a given type - a 3-month non-compete and a 5-year non-compete both got flagged "HIGH — check the duration" with identical wording, regardless of what the clause actually says. Model 3 replaces that with an LLM that reads the actual clause text and grounds its explanation in the specific terms present (durations, amounts, conditions).

# Cell 1 - Install Dependencies

In [ ]:
!pip install groq pandas -q

# Cell 2 - Import Libraries

In [ ]:
import json
import re
import time
import pandas as pd

# Cell 3 - Set Up the LLM Client (Groq API)

Enter your Groq API key when prompted (get one at console.groq.com/keys). If you skip this, the notebook runs in **mock mode** instead -- using a deterministic stand-in explanation generator so the whole pipeline still runs end-to-end without a key.

In [ ]:
import getpass

client = None
MODEL_NAME = "openai/gpt-oss-120b"  # Groq deprecates/renames models fairly often --
                                     # check console.groq.com/docs/models if this 404s

try:
    api_key = getpass.getpass("Groq API key (leave blank to run in mock mode): ")
    if api_key.strip():
        import groq
        client = groq.Groq(api_key=api_key.strip())
        print("Client configured -- using live API calls.")
    else:
        print("No key entered -- running in MOCK MODE (deterministic stand-in explanations).")
except Exception as e:
    print(f"Falling back to mock mode ({e}).")

# Cell 4 - Load Model 2's Classified Clauses and Apply the Confidence Threshold

In the full pipeline this DataFrame comes directly from Model 2's `classify_clauses()` output (columns: `clause_text`, `predicted_label`, `confidence`, ...). For standalone testing here, if no CSV is uploaded, a small built-in sample (Model 2's own custom test clauses, plus 2 deliberately out-of-scope ones) is used instead.

**New after real-contract testing:** a real MOM contract template run through the full pipeline showed the classifier confidently mislabelling administrative content (commencement dates, signature blocks) as one of the 6 tracked clause types, simply because it has no "none of these" option. On that real run, all 7 wrong predictions had confidence <= 35%, and all 4 correct ones were >= 35% -- so this cell now relabels anything below that threshold as `unclassified` instead of providing the wrong answer.

In [ ]:
from google.colab import files

def load_model2_output():
    print("Upload Model 2's classified-clauses CSV, or skip to use a built-in sample instead.")
    try:
        uploaded = files.upload()
        if uploaded:
            path = list(uploaded.keys())[0]
            print(f"Using uploaded file: {path}")
            return pd.read_csv(path)
    except Exception as e:
        print(f"No file uploaded ({e}). Using built-in sample instead.")

    sample = [
        {"clause_text": "The Employee shall receive a monthly salary of SGD 5,000, payable on the last working day of each month, together with an annual performance bonus at the discretion of the Company.",
         "predicted_label": "compensation clause", "confidence": 0.81},
        {"clause_text": "Either party may terminate this Agreement by giving one month's written notice. The Company reserves the right to terminate employment immediately for gross misconduct without notice or compensation.",
         "predicted_label": "termination clause", "confidence": 0.77},
        {"clause_text": "The Employee shall not, during the term of employment and for a period of 12 months thereafter, directly or indirectly engage in any business that competes with the Company within Singapore.",
         "predicted_label": "non-compete clause", "confidence": 0.85},
        {"clause_text": "The Employee shall not, during the term of employment and for a period of 5 years thereafter, directly or indirectly engage in any business anywhere in the world that competes with the Company.",
         "predicted_label": "non-compete clause", "confidence": 0.79},
        {"clause_text": "The Employee agrees to keep confidential all trade secrets, business strategies, client information, and proprietary data obtained during employment and shall not disclose such information to any third party.",
         "predicted_label": "confidentiality clause", "confidence": 0.74},
        {"clause_text": "The Employee's first three months of employment shall be a probationary period, during which either party may terminate this contract with one week's notice.",
         "predicted_label": "probation clause", "confidence": 0.70},
        {"clause_text": "All inventions, developments, software, and intellectual property created by the Employee in the course of employment shall be the exclusive property of the Company, whether or not created during working hours.",
         "predicted_label": "intellectual property clause", "confidence": 0.83},
        {"clause_text": "Your employment will commence on the date specified above. Place of work: Assignment site as notified.",
         "predicted_label": "probation clause", "confidence": 0.35},
        {"clause_text": "Sample Employment Contract Updated on 02/12/2011 12:37:31",
         "predicted_label": "intellectual property clause", "confidence": 0.21},
    ]
    print("Using built-in sample clauses (9 clauses, includes 2 non-compete durations for "
          "comparison, plus 2 low-confidence out-of-scope clauses to demonstrate the "
          "confidence-threshold handling below).")
    return pd.DataFrame(sample)


results_df = load_model2_output()
print(f"\nLoaded {len(results_df)} classified clauses")

# --- Confidence threshold: relabel low-confidence predictions as 'unclassified' ---
# Added after real-contract testing showed the classifier has no way to say "none
# of the 6 types fit" -- it always forces a best guess among the 6, even for
# administrative/procedural clauses (commencement dates, signature blocks, working
# hours) that don't belong to any of them. On a real test run, the 7 genuinely
# wrong predictions clustered at confidence <= 0.35, while all 4 correct ones
# were >= 0.35 -- so clauses below this threshold are relabelled 'unclassified'
# instead of a confident-looking wrong answer.
CONFIDENCE_THRESHOLD = 0.35

def apply_confidence_threshold(df, threshold=CONFIDENCE_THRESHOLD):
    df = df.copy()
    below = df['confidence'] < threshold
    df.loc[below, 'predicted_label'] = 'unclassified'
    return df

n_before = (results_df['predicted_label'] != 'unclassified').sum()
results_df = apply_confidence_threshold(results_df)
n_reclassified = (results_df['predicted_label'] == 'unclassified').sum()
print(f"{n_reclassified} / {len(results_df)} clauses fell below {CONFIDENCE_THRESHOLD:.0%} "
      f"confidence and were relabelled 'unclassified' instead of a forced guess")

results_df[['clause_text', 'predicted_label', 'confidence']].head()

# Cell 5 - Define the Explanation Prompt Template

Asks the LLM to return strict JSON so downstream code can parse it reliably and explicitly instructs it to ground the risk level and explanation in the specific wording of the clause instead of giving a generic answer for the clause type.

**Constraint worth stating explicitly:** the prompt tells the model the clause's type has already been decided by Model 2 and instructs it not to reconsider that classification -- Model 3's role here is explanation and risk assessment only, never a second classification opinion.

In [ ]:
PROMPT_TEMPLATE = """You are helping a non-lawyer employee in Singapore understand one clause \
from their employment contract. You will be given the clause's type (as classified by an \
earlier model, which may occasionally be wrong) and its exact text.

Clause type: {clause_type}
Clause text: "{clause_text}"

Respond with ONLY a valid JSON object (no other text, no markdown fences) with exactly these keys:
- "risk_level": one of "LOW", "MEDIUM", "HIGH" -- based on the ACTUAL terms in this specific \
clause (e.g. a 24-month non-compete is higher risk than a 3-month one; a vague, broad \
confidentiality definition is higher risk than a narrow one).
- "explanation": one or two plain-English sentences explaining what this specific clause means \
for the employee, referencing concrete details from the text (durations, amounts, conditions) \
where present.
- "watch_for": one short, concrete thing the employee should check or ask about, specific to \
what's actually written here -- not a generic template answer.
"""


def build_prompt(clause_text, predicted_label):
    return PROMPT_TEMPLATE.format(clause_type=predicted_label, clause_text=clause_text)

# Cell 6 - Explanation Generation Function (calls the LLM, parses JSON, handles failures)

Two genuinely different situations, handled differently and honestly:
1. **No client configured at all** (blank key) -> clearly-labelled mock mode, using a deterministic stand-in so the notebook still runs end-to-end without an API key.
2. **A real client exists but a live call fails, or returns unparseable output after retrying** -> this is a genuine error and is surfaced honestly as an `ERROR` risk level distinct from a real assessment. An earlier version of this cell silently substituted mock output or Model 2's static `RISK_MAP` here, which disguises a real failure as a plausible-looking answer -- the same category of issue caught and fixed in the deployed web application. That design is not repeated here.

JSON is extracted by locating the response's first `{` and last `}` instead of assuming the whole response is clean JSON since reasoning-tuned models sometimes prepend brief explanatory text before their structured answer.

In [ ]:
# Kept only as the mock-mode stand-in's source of plausible type-based text --
# never used to disguise a real live-API failure (see generate_explanation below).
RISK_MAP = {
    "non-compete clause": {"level": "HIGH", "reason": "This clause restricts your ability to work for competitors after leaving. Check the duration and geographic scope carefully."},
    "intellectual property clause": {"level": "HIGH", "reason": "This may assign ownership of your personal projects or inventions to your employer. Check if it covers work done outside office hours."},
    "termination clause": {"level": "MEDIUM", "reason": "This defines how employment can be ended. Check notice periods and conditions for immediate termination without pay."},
    "confidentiality clause": {"level": "MEDIUM", "reason": "This restricts what you can discuss outside work. Check how broadly 'confidential information' is defined."},
    "compensation clause": {"level": "LOW", "reason": "This defines your pay and benefits. Verify the figures match what was discussed during your interview."},
    "probation clause": {"level": "LOW", "reason": "This sets the trial period terms. Check the duration and what happens at the end of probation."},
}


def _mock_llm_call(clause_text, predicted_label):
    """Deterministic stand-in used only when no API key was ever provided."""
    base = RISK_MAP.get(predicted_label, {"level": "MEDIUM", "reason": "Review this clause carefully."})
    numbers = re.findall(r'\d+\s*(?:month|day|week|year)s?', clause_text, flags=re.I)
    detail = f" (note: this clause specifies {', '.join(numbers)})" if numbers else ""
    return json.dumps({
        "risk_level": base["level"],
        "explanation": base["reason"] + detail,
        "watch_for": "Confirm this matches what was verbally agreed during your offer discussion."
    })


def generate_explanation(clause_text, predicted_label, client=None, model=MODEL_NAME, max_retries=2):
    if predicted_label == 'unclassified':
        return {
            "risk_level": "N/A",
            "explanation": "This clause didn't clearly match any of the 6 tracked categories "
                            "(compensation, termination, confidentiality, non-compete, IP, or "
                            "probation) -- it may be administrative or procedural content.",
            "watch_for": "Skim this manually if it looks important; automatic risk assessment "
                         "wasn't confident enough to be reliable here."
        }

    # No key was ever provided -- clearly-labelled mock mode, not a failure.
    if client is None:
        raw = _mock_llm_call(clause_text, predicted_label)
        return json.loads(raw)

    # A real client exists -- any failure from here is a genuine error and must be
    # surfaced honestly and not silently disguised as mock output or a generic
    # type-based guess.
    prompt = build_prompt(clause_text, predicted_label)
    raw = None
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model, max_tokens=300,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = resp.choices[0].message.content
            break
        except Exception as e:
            last_error = e
            print(f"  [generate_explanation] API call failed (attempt {attempt + 1}/"
                  f"{max_retries + 1}): {type(e).__name__}: {e}")
            if attempt < max_retries:
                time.sleep(1)

    error_state = {
        "risk_level": "ERROR",
        "explanation": "This clause was classified but couldn't be explained automatically "
                        f"due to a connection issue "
                        f"({type(last_error).__name__ if last_error else 'unknown'}).",
        "watch_for": "Review this clause manually."
    }
    if raw is None:
        return error_state

    try:
        # Locate the JSON object by its first '{' and last '}' instead of assuming
        # the whole response is clean JSON -- necessary because reasoning-tuned models
        # sometimes prepend brief explanatory text before the structured answer.
        start, end = raw.find('{'), raw.rfind('}')
        if start == -1 or end == -1 or end < start:
            raise ValueError("No JSON object found in response")
        parsed = json.loads(raw[start:end + 1])
        assert parsed.get("risk_level") in ("LOW", "MEDIUM", "HIGH")
        assert parsed.get("explanation")
        assert parsed.get("watch_for")
        return parsed
    except Exception as e:
        print(f"  [generate_explanation] Response wasn't valid JSON: {e}\nRaw: {raw[:200]}")
        return error_state


# Cell 7 - Run Explanation Generation Over All Classified Clauses

In [ ]:
explanations = []
print(f"Generating explanations for {len(results_df)} clauses "
      f"({'LIVE API' if client else 'MOCK MODE'})...\n")

for i, row in results_df.iterrows():
    result = generate_explanation(row['clause_text'], row['predicted_label'], client=client)
    explanations.append(result)
    print(f"  [{i+1}/{len(results_df)}] {row['predicted_label']} -> {result['risk_level']}")

explained_df = results_df.copy()
explained_df['risk_level'] = [e['risk_level'] for e in explanations]
explained_df['explanation'] = [e['explanation'] for e in explanations]
explained_df['watch_for'] = [e['watch_for'] for e in explanations]

print("\nDone.")

# Cell 8 - Validate Output Quality (checks structure and flags parsing failures)

Checks worth running before trusting this output: did every clause get a valid risk level and what the specific thing Model 3 exists to fix - do clauses of the *same type* actually get *different* explanations when their specific wording differs?

In [ ]:
print("=" * 60)
print("MODEL 3 OUTPUT VALIDATION")
print("=" * 60)
print(f"Total explained:     {len(explained_df)}")
print(f"Risk level counts:\n{explained_df['risk_level'].value_counts().to_string()}")

invalid = explained_df[~explained_df['risk_level'].isin(['LOW', 'MEDIUM', 'HIGH', 'N/A'])]
print(f"\nInvalid risk levels: {len(invalid)} (should be 0; 'N/A' is expected for unclassified clauses)")

n_unclassified = (explained_df['predicted_label'] == 'unclassified').sum()
print(f"Unclassified clauses (below confidence threshold): {n_unclassified}")

# Specific check: do the two non-compete clauses (if present) get DIFFERENT
# explanations, given they specify very different durations?
noncompete = explained_df[explained_df['predicted_label'] == 'non-compete clause']
if len(noncompete) >= 2:
    unique_explanations = noncompete['explanation'].nunique()
    status = "PASS" if unique_explanations > 1 else "CHECK"
    print(f"\n[{status}] Non-compete clauses differentiated by content: "
          f"{unique_explanations}/{len(noncompete)} unique explanations")
    for _, row in noncompete.iterrows():
        print(f"    - {row['explanation']}")
print("=" * 60)

# Cell 9 - Generate the Final User-Facing Report

In [ ]:
def generate_report(df, n=None):
    n = n or len(df)
    print("=" * 60)
    print("CONTRACT ANALYSIS AI — Employment Contract Review")
    print("=" * 60)
    print("DISCLAIMER: This tool is for informational purposes only")
    print("and does not constitute legal advice.")
    print("=" * 60)

    for i, row in df.head(n).iterrows():
        print(f"\nCLAUSE {i+1}: {row['predicted_label'].upper()}")
        print(f"  Risk Level:  {row['risk_level']}")
        print(f"  Text:        {row['clause_text'][:150]}{'...' if len(row['clause_text'])>150 else ''}")
        print(f"  Explanation: {row['explanation']}")
        print(f"  Watch for:   {row['watch_for']}")
        if 'confidence' in row:
            print(f"  Confidence:  {row['confidence']:.0%}")


generate_report(explained_df)

# Cell 10 - Save Results to File

In [ ]:
output_path = "model3_explained_clauses.csv"
explained_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

try:
    files.download(output_path)
except Exception as e:
    print(f"(Download skipped — not running in Colab: {e})")

# Cell 11 - Summary of Results

In [ ]:
print("=" * 60)
print("MODEL 3 SUMMARY -- LLM-BASED EXPLANATION GENERATION")
print("=" * 60)
print(f"Clauses explained:  {len(explained_df)}")
print(f"Mode:               {'LIVE API (' + MODEL_NAME + ')' if client else 'MOCK (no API key provided)'}")

print("\n--- What this notebook demonstrates ---")
print("  1. Clause-specific explanations grounded in actual clause wording,")
print("     not a generic per-type template like Model 2's RISK_MAP alone")
print("  2. Strict JSON prompting with robust parsing -- locates the response's")
print("     first '{' and last '}' rather than assuming clean JSON, since")
print("     reasoning-tuned models sometimes prepend explanatory text first")
print("  3. Honest failure handling: a missing API key runs in clearly-labelled")
print("     mock mode; a genuine live failure returns an ERROR state distinct")
print("     from a real assessment, never a disguised guess")
print("  4. Validated that same-type clauses with different real terms")
print("     (12-month vs 5-year non-compete) get differentiated explanations")
print("  5. Confidence threshold (35%) relabels administrative/out-of-scope")
print("     clauses as 'unclassified' instead of a confidently wrong guess --")
print("     found necessary after a real contract test showed 7/7 wrong")
print("     predictions clustered below this threshold, all 4 correct ones above it")

print("\n--- Known limitations ---")
print("  - Mock mode explanations are still fairly generic; only the live")
print("    API path gives genuinely clause-specific reasoning")
print("  - No cost/rate-limit handling for large contracts (many clauses ->")
print("    many API calls); worth batching or caching for a real deployment")
print("  - Risk-level judgement is the LLM's own assessment, not validated")
print("    against real legal expertise -- frame this clearly as informational,")
print("    not legal advice, in the final report/UI")
print("  - Risk-level judgements are not perfectly deterministic between runs;")
print("    a clause near the boundary between two risk levels may be judged")
print("    differently on separate runs -- see the report's Evaluation chapter")

print("\n--- Status (updated since the preliminary report) ---")
print("  - LEGAL-BERT fine-tuning (Model 2) is complete, not a future step:")
print("    see Model2_LegalBERT_FineTuning.ipynb for real, held-out CUAD")
print("    results -- a 33-68 percentage-point F1 gain over zero-shot on")
print("    4 of 6 categories. It is validated but not yet deployed; the")
print("    report's Design chapter explains why.")
print("  - This notebook, the full Pipeline_End_to_End.ipynb, and the")
print("    deployed web application all now run against real, live Groq")
print("    API calls, not just mock output.")
print("=" * 60)
